# 01 — Integración: construcción de Base 1 (`timeline_df`)

**HealthSignal LATAM — RISA_DATA_V1.0**

## ¿Qué hace este notebook, en palabras simples?

RISA entrega los datos de cada paciente repartidos en varios archivos (signos vitales,
wearable, dispositivos médicos, laboratorio), cada uno con su propia forma, su propia
frecuencia de muestreo y sus propios problemas de calidad. Este notebook toma las **4**
tablas que registran *lecturas puntuales* (algo que se midió en un instante exacto) y las
apila en una sola tabla larga llamada `timeline_df` — **Base 1**.

No es un cruce/join: es un apilamiento (`UNION`). Cada fila sigue siendo "una lectura",
solo que ahora todas viven en la misma tabla, con las mismas columnas, para poder
procesarlas juntas más adelante (Base 2, en el notebook `02_features_ventanas.ipynb`).

**Las 4 tablas que se apilan aquí (todas tienen un timestamp de un solo instante, no un rango):**
- `vital_signs` — HR, RR, SpO2, TEMP, SBP, DBP (monitores clínicos)
- `wearable_observations` — WEARABLE_HR, STEPS, ACTIVITY_LEVEL (pulsera)
- `device_observations` — SIGNAL_QUALITY_INDEX (calidad de señal del dispositivo)
- `laboratory_results` — resultados de laboratorio

**Lo que NO entra aquí** (y por qué): `medication_administrations`, `encounters`,
`patient_context` y `connectivity_events` tienen un *inicio y un fin* (duran un rango de
tiempo, no son un instante) — esas se usan en el notebook 02 como contexto, unidas por
solape de fechas, no apiladas aquí. `conditions` tampoco entra: es historial de
comorbilidad del paciente (estático), no una medición con fecha.

## Antes de correr esto

Corre las celdas en orden (`Kernel → Restart & Run All`), con los 4 archivos reales
completos en `data/raw/` (`vital_signs.csv`, `wearable_observations.csv`,
`device_observations.csv`, `laboratory_results.csv`). Al final vas a tener un archivo
`../data/processed/timeline_df.csv` con TODAS las lecturas de los ~1000 pacientes, listas
para la siguiente etapa (`02_features_ventanas.ipynb`).

> La lógica de este notebook (parseo de fechas, conversión de unidades, chequeo de
> plausibilidad, deduplicación, unión) ya se validó celda por celda contra tus datos reales
> chicos (`device_observations`, `laboratory_results`) y contra casos de prueba diseñados
> para activar cada regla — está lista para correr contra el dataset completo tal cual.
> `vital_signs` (1,622,969 filas) y `wearable_observations` (895,551 filas) van a tardar más
> en cargar que las tablas chicas, es normal — dale unos minutos.


## Paso 0 — Preparar el entorno

Importamos `pandas` (para tablas) y `numpy` (para cálculos numéricos), y definimos dónde
están los datos crudos. `DATA_DIR` apunta a `../data/raw` porque este notebook vive en la
carpeta `notebooks/`, que está al lado de `data/` en el repo.

In [1]:
import pandas as pd
import numpy as np
import os

# DATA_DIR apunta a la carpeta de datos crudos del repo
# (este notebook vive en notebooks/, que está al lado de data/)
DATA_DIR = "../data/raw"
print("Leyendo datos crudos desde:", DATA_DIR)

Leyendo datos crudos desde: ../data/raw


## Paso 1 — Cargar las 4 tablas y parsear las fechas correctamente

Los timestamps de RISA vienen en **formato mixto**: algunas filas traen microsegundos
(`2026-07-16 00:44:56.969159`) y otras no (`2026-07-16 00:44:56`). Si usamos el parser de
fechas por defecto de pandas, las filas con el formato "raro" se pueden convertir en `NaT`
(fecha vacía) **sin avisar** — y se pierden en silencio. Por eso usamos siempre
`format="mixed"`, que entiende ambos formatos a la vez.

Esto no es un detalle menor: en `laboratory_results` son justo las filas con formato distinto
las que tienen los resultados fuera de rango más importantes — un parser descuidado perdería
exactamente los datos que más importan.

Después de cargar, comprobamos con un `assert` que ninguna fecha quedó vacía. Si algo falla
aquí, el notebook se detiene en seco en vez de seguir con datos corruptos silenciosamente.

In [2]:
vital_signs = pd.read_csv(f"{DATA_DIR}/03_monitoring/vital_signs.csv")
vital_signs["timestamp"] = pd.to_datetime(vital_signs["timestamp"], format="mixed")

wearable_observations = pd.read_csv(f"{DATA_DIR}/03_monitoring/wearable_observations.csv")
wearable_observations["timestamp"] = pd.to_datetime(wearable_observations["timestamp"], format="mixed")
wearable_observations["sync_datetime"] = pd.to_datetime(wearable_observations["sync_datetime"], format="mixed")

device_observations = pd.read_csv(f"{DATA_DIR}/03_monitoring/device_observations.csv")
device_observations["timestamp"] = pd.to_datetime(device_observations["timestamp"], format="mixed")

laboratory_results = pd.read_csv(f"{DATA_DIR}/02_clinical/laboratory_results.csv")
laboratory_results["sample_datetime"] = pd.to_datetime(laboratory_results["sample_datetime"], format="mixed")
laboratory_results["result_datetime"] = pd.to_datetime(laboratory_results["result_datetime"], format="mixed")

print("Filas cargadas:")
print("  vital_signs           :", len(vital_signs))
print("  wearable_observations :", len(wearable_observations))
print("  device_observations   :", len(device_observations))
print("  laboratory_results    :", len(laboratory_results))

# Si algo se volvió NaT, mejor enterarnos ahora que más adelante
assert vital_signs["timestamp"].isna().sum() == 0, "Hay timestamps de vital_signs que quedaron vacíos (NaT)"
assert wearable_observations["timestamp"].isna().sum() == 0, "Hay timestamps de wearable que quedaron vacíos (NaT)"
assert device_observations["timestamp"].isna().sum() == 0, "Hay timestamps de device_observations que quedaron vacíos (NaT)"
assert laboratory_results["sample_datetime"].isna().sum() == 0, "Hay sample_datetime que quedaron vacíos (NaT) -- revisar el parseo de fechas"
assert laboratory_results["result_datetime"].isna().sum() == 0, "Hay result_datetime que quedaron vacíos (NaT)"
print("\nTodas las fechas se parsearon correctamente (0 valores vacíos).")

Filas cargadas:
  vital_signs           : 1622969
  wearable_observations : 895551
  device_observations   : 13329
  laboratory_results    : 4593

Todas las fechas se parsearon correctamente (0 valores vacíos).


## Paso 2 — Convertir el valor de texto a número en `wearable_observations`

En `wearable_observations`, la columna `value` viene guardada como **texto** (`"79.0"` en vez
de `79.0`), incluso para las variables numéricas. Si no se convierte, cualquier cálculo
posterior (promedio, z-score) fallaría o daría resultados sin sentido.

Ojo: no todas las variables de esta tabla son numéricas — `ACTIVITY_LEVEL` es una categoría
de texto (`"LOW"`, `"HIGH"`, etc.), no un número. Por eso solo convertimos `WEARABLE_HR` y
`STEPS`, y dejamos las categóricas tal como están.

In [3]:
VARIABLES_NUMERICAS_WEARABLE = {"WEARABLE_HR", "STEPS"}

es_numerica = wearable_observations["variable_code"].isin(VARIABLES_NUMERICAS_WEARABLE)
valor_convertido = pd.to_numeric(wearable_observations["value"].where(es_numerica), errors="coerce")

# value_final: el número convertido para las variables numéricas, el texto original para las categóricas
wearable_observations["value_final"] = valor_convertido.where(es_numerica, wearable_observations["value"])

print("Ejemplo de conversión (antes eran texto, ahora son número donde corresponde):")
wearable_observations[["variable_code", "value", "value_final"]].head(6)

Ejemplo de conversión (antes eran texto, ahora son número donde corresponde):


,variable_code,value,value_final
0,WEARABLE_HR,70.194,70.194
1,STEPS,0,0.0
2,ACTIVITY_LEVEL,REST,REST
3,WEARABLE_HR,73.361,73.361
4,STEPS,0,0.0
5,ACTIVITY_LEVEL,REST,REST


## Paso 3 — Normalizar unidades contra `units_catalog`

No todos los dispositivos reportan en la misma unidad — por ejemplo, algunas lecturas de
`TEMP` vienen en grados Fahrenheit en vez de Celsius. `units_catalog.csv` (en `05_metadata`)
trae, para cada unidad, a qué unidad canónica convertirla y con qué fórmula:

```
valor_canónico = valor_original × conversion_factor + conversion_offset
```

**Esto se aplica a TODAS las filas, no solo a las que vienen "marcadas" como en unidad
rara** — confiar en que el dato venga bien marcado es exactamente el tipo de suposición
que este dataset está diseñado para castigar.

In [4]:
units_catalog = pd.read_csv(f"{DATA_DIR}/05_metadata/units_catalog.csv")
units_catalog = units_catalog.rename(columns={"unit_code": "unit"})

def normalizar_unidades(df, value_col, unit_col="unit"):
    # Convierte value_col a la unidad canónica de units_catalog, siempre (no solo si viene marcado).
    merged = df.merge(
        units_catalog[["unit", "canonical_unit", "conversion_factor", "conversion_offset"]],
        left_on=unit_col, right_on="unit", how="left"
    )
    valor_num = pd.to_numeric(merged[value_col], errors="coerce")
    convertido = valor_num * merged["conversion_factor"].fillna(1) + merged["conversion_offset"].fillna(0)
    # si la variable es categórica (no tiene unidad numérica en el catálogo), se deja el valor tal cual
    merged[value_col] = convertido.where(merged["canonical_unit"].notna(), merged[value_col])
    merged[unit_col] = merged["canonical_unit"].fillna(merged[unit_col])
    return merged.drop(columns=["canonical_unit", "conversion_factor", "conversion_offset"])

vital_signs = normalizar_unidades(vital_signs, "value")
device_observations = normalizar_unidades(device_observations, "value")
wearable_observations = normalizar_unidades(wearable_observations, "value_final")
laboratory_results = normalizar_unidades(laboratory_results, "result_value")

# Chequeo visual: si hay algún TEMP en grados Fahrenheit en la muestra, debe haber quedado en Celsius
temp_rows = vital_signs[vital_signs["variable_code"] == "TEMP"]
print("Filas de TEMP tras normalizar unidades (deben quedar todas en degC, valores 30-45):")
print(temp_rows[["variable_code", "value", "unit"]])

Filas de TEMP tras normalizar unidades (deben quedar todas en degC, valores 30-45):
        variable_code   value  unit
1923             TEMP  36.764  degC
1924             TEMP  36.569  degC
1925             TEMP  36.732  degC
1926             TEMP  36.659  degC
1927             TEMP  36.808  degC
...               ...     ...   ...
1622327          TEMP  36.870  degC
1622328          TEMP  36.726  degC
1622329          TEMP  36.863  degC
1622330          TEMP  36.870  degC
1622331          TEMP  37.021  degC

[147949 rows x 3 columns]


## Paso 4 — Marcar valores implausibles, sin confiar en `quality_flag`

El EDA encontró 549 lecturas de SpO2 por encima de 100% marcadas como `quality_flag = "OK"`
por el sistema de origen — el dato viene "certificado" como bueno y aun así es físicamente
imposible. Por eso este chequeo se hace de forma **independiente** de `quality_flag`,
comparando cada valor contra el rango físico plausible de `variable_catalog.csv`
(`plausibility_min` / `plausibility_max`).

El resultado es una columna nueva, `plausible` (True/False) — **no se borra ninguna fila**,
solo se marca. Borrar destruiría evidencia que después puede hacer falta para explicar una
señal o auditar el dato.

In [5]:
variable_catalog = pd.read_csv(f"{DATA_DIR}/05_metadata/variable_catalog.csv")
rango_plausible = variable_catalog.set_index("variable_code")[["plausibility_min", "plausibility_max"]]

def marcar_plausibilidad(df, value_col, var_col="variable_code"):
    merged = df.merge(rango_plausible, left_on=var_col, right_index=True, how="left")
    valor_num = pd.to_numeric(merged[value_col], errors="coerce")
    dentro_de_rango = valor_num.between(merged["plausibility_min"], merged["plausibility_max"])
    sin_rango_definido = merged["plausibility_min"].isna()  # variables categóricas (ACTIVITY_LEVEL, etc.)
    merged["plausible"] = dentro_de_rango | sin_rango_definido
    return merged.drop(columns=["plausibility_min", "plausibility_max"])

vital_signs = marcar_plausibilidad(vital_signs, "value")
device_observations = marcar_plausibilidad(device_observations, "value")
wearable_observations = marcar_plausibilidad(wearable_observations, "value_final")
laboratory_results = marcar_plausibilidad(laboratory_results, "result_value", var_col="test_code")

implausibles = vital_signs[~vital_signs["plausible"]]
print(f"vital_signs: {len(implausibles)} filas marcadas como NO plausibles (aunque su quality_flag pueda decir OK)")
implausibles[["variable_code", "value", "quality_flag", "plausible"]]

vital_signs: 762 filas marcadas como NO plausibles (aunque su quality_flag pueda decir OK)


,variable_code,value,quality_flag,plausible
29673,SpO2,103.185,CHECK,False
32377,SpO2,100.150,CHECK,False
34571,SpO2,103.298,CHECK,False
48880,RR,3.247,CHECK,False
60188,RR,0.836,CHECK,False
...,...,...,...,...
1611511,SpO2,100.105,OK,False
1611541,SpO2,100.209,OK,False
1611574,SpO2,100.561,OK,False
1611675,SpO2,100.269,OK,False


## Paso 5 — Deduplicar retransmisiones en `vital_signs`

El EDA encontró ~540 pares de filas duplicadas por **retransmisión** del monitor
(`source_system = "MONITOR_RETRANSMIT"`): la misma lectura (mismo paciente, misma variable,
mismo instante, mismo valor) llega dos veces porque el dispositivo la reenvió. Si no se
deduplica antes de construir la serie de tiempo, esa lectura pesa el doble en cualquier
promedio o conteo.

La regla: cuando hay dos filas idénticas en (`patient_id`, `variable_code`, `timestamp`,
`value`), nos quedamos con la que NO vino marcada como retransmisión.

In [6]:
def deduplicar_retransmisiones(df, subset):
    df = df.copy()
    # prioridad 0 = nos la quedamos primero; 1 = solo si no hay otra opción
    df["_prioridad"] = (df["source_system"] == "MONITOR_RETRANSMIT").astype(int)
    df = df.sort_values("_prioridad")
    antes = len(df)
    df = df.drop_duplicates(subset=subset, keep="first").drop(columns="_prioridad")
    print(f"vital_signs: {antes} -> {len(df)} filas ({antes - len(df)} duplicados de retransmisión eliminados)")
    return df

vital_signs = deduplicar_retransmisiones(
    vital_signs, subset=["patient_id", "variable_code", "timestamp", "value"]
)

vital_signs: 1622969 -> 1622429 filas (540 duplicados de retransmisión eliminados)


## Paso 6 — Llevar las 4 tablas a un esquema común

Antes de poder apilarlas, las 4 tablas necesitan tener exactamente las mismas columnas, con
el mismo significado. La función de abajo toma cada tabla y arma esas columnas:

| columna común | qué significa |
|---|---|
| `patient_id` | paciente |
| `timestamp` | cuándo ocurrió la medición |
| `available_datetime` | cuándo ese dato **existió** para el sistema (importante para no usar información "del futuro" — en laboratorio es `result_datetime`, no `sample_datetime`) |
| `variable_code` | qué se midió (HR, TEMP, WEARABLE_HR, etc. — en labs viene en la columna `test_code`) |
| `value` | el valor ya limpio (unidad normalizada) |
| `unit` | unidad canónica |
| `quality_flag` | el indicador de calidad de la fuente original |
| `plausible` | si el valor pasó el chequeo de plausibilidad del Paso 4 |
| `source_table` | de qué tabla original vino (para poder rastrearla después) |
| `record_id` | el ID original de la fila (para trazabilidad — `evidence.csv` lo va a necesitar) |

In [7]:
ESQUEMA_COMUN = ["patient_id", "timestamp", "available_datetime", "variable_code",
                  "value", "unit", "quality_flag", "plausible", "source_table", "record_id"]

def a_esquema_comun(df, record_id_col, ts_col="timestamp", available_col=None,
                     value_col="value", quality_col="quality_flag", var_col="variable_code",
                     source_table=""):
    out = pd.DataFrame()
    out["patient_id"] = df["patient_id"]
    out["timestamp"] = df[ts_col]
    out["available_datetime"] = df[available_col] if available_col else df[ts_col]
    out["variable_code"] = df[var_col]
    out["value"] = df[value_col]
    out["unit"] = df["unit"]
    out["quality_flag"] = df[quality_col] if quality_col in df.columns else np.nan
    out["plausible"] = df["plausible"] if "plausible" in df.columns else np.nan
    out["source_table"] = source_table
    out["record_id"] = df[record_id_col]
    return out[ESQUEMA_COMUN]

vs_common = a_esquema_comun(vital_signs, "observation_id",
                             source_table="vital_signs")

wo_common = a_esquema_comun(wearable_observations, "wearable_observation_id",
                             value_col="value_final", quality_col="measurement_quality",
                             source_table="wearable_observations")

do_common = a_esquema_comun(device_observations, "device_observation_id",
                             quality_col="signal_quality",
                             source_table="device_observations")

lr_common = a_esquema_comun(laboratory_results, "lab_result_id",
                             ts_col="sample_datetime", available_col="result_datetime",
                             value_col="result_value", quality_col="quality_flag",
                             var_col="test_code", source_table="laboratory_results")

print("Filas por tabla ya en esquema común:")
for nombre, df in [("vital_signs", vs_common), ("wearable_observations", wo_common),
                    ("device_observations", do_common), ("laboratory_results", lr_common)]:
    print(f"  {nombre:24s}: {len(df)}")

Filas por tabla ya en esquema común:
  vital_signs             : 1622429
  wearable_observations   : 895551
  device_observations     : 13329
  laboratory_results      : 4593


## Paso 7 — Unir (apilar) las 4 tablas en `timeline_df`

`pd.concat` apila las tablas una debajo de otra (no las cruza) — es Base 1 completa.

In [8]:
timeline_df = pd.concat([vs_common, wo_common, do_common, lr_common], ignore_index=True)
timeline_df = timeline_df.sort_values(["patient_id", "timestamp"]).reset_index(drop=True)

print("timeline_df -- filas totales:", len(timeline_df))
print()
print(timeline_df["source_table"].value_counts())
print()
timeline_df.head(10)

timeline_df -- filas totales: 2535902

source_table
vital_signs              1622429
wearable_observations     895551
device_observations        13329
laboratory_results          4593
Name: count, dtype: int64



,patient_id,timestamp,available_datetime,variable_code,value,unit,quality_flag,plausible,source_table,record_id
0,PAT-0001,2026-07-10 09:00:00,2026-07-10 09:00:00,HR,74.346,bpm,OK,True,vital_signs,OBS-0000000001
1,PAT-0001,2026-07-10 09:00:00,2026-07-10 09:00:00,RR,13.106,rpm,OK,True,vital_signs,OBS-0000000642
2,PAT-0001,2026-07-10 09:00:00,2026-07-10 09:00:00,SpO2,95.989,%,OK,True,vital_signs,OBS-0000001286
3,PAT-0001,2026-07-10 09:00:00,2026-07-10 09:00:00,TEMP,36.764,degC,OK,True,vital_signs,OBS-0000001924
4,PAT-0001,2026-07-10 09:00:00,2026-07-10 09:00:00,DBP,78.226,mmHg,OK,True,vital_signs,OBS-0000002247
5,PAT-0001,2026-07-10 09:00:00,2026-07-10 09:00:00,WEARABLE_HR,70.194,bpm,OK,True,wearable_observations,WOBS-0000000001
6,PAT-0001,2026-07-10 09:00:00,2026-07-10 09:00:00,STEPS,0.000,count,OK,True,wearable_observations,WOBS-0000000002
7,PAT-0001,2026-07-10 09:00:00,2026-07-10 09:00:00,ACTIVITY_LEVEL,NaN,category,OK,True,wearable_observations,WOBS-0000000003
8,PAT-0001,2026-07-10 09:00:00,2026-07-10 09:00:00,SIGNAL_QUALITY_INDEX,0.930,ratio,0.93,True,device_observations,DOBS-000000001
9,PAT-0001,2026-07-10 09:20:00,2026-07-10 09:20:00,HR,71.325,bpm,OK,True,vital_signs,OBS-0000000002


## Paso 8 — Validar antes de guardar (anti-leakage y completitud)

Dos chequeos mínimos que no pueden fallar:

1. **Anti-leakage**: `available_datetime` nunca puede ser anterior a `timestamp` — sería
   decir que un dato "existió" antes de haber ocurrido, lo cual es imposible y es exactamente
   lo que el validador oficial de la entrega (`validate_submission.py`) revisa más adelante
   en `evidence.csv`.
2. **Completitud**: ninguna fila puede quedar sin `patient_id` o sin `timestamp`.

Si algo de esto falla, es mejor que el notebook se detenga aquí a que seguir con datos
rotos hasta el final.

In [9]:
assert (timeline_df["available_datetime"] >= timeline_df["timestamp"]).all(), \
    "Hay filas donde available_datetime es ANTERIOR al evento -- revisar el mapeo de columnas"
assert timeline_df["patient_id"].notna().all(), "Hay filas sin patient_id"
assert timeline_df["timestamp"].notna().all(), "Hay filas sin timestamp"

print("Validaciones OK:")
print(f"  - available_datetime >= timestamp en el 100% de las filas")
print(f"  - 0 filas sin patient_id, 0 filas sin timestamp")
print(f"  - {(~timeline_df['plausible']).sum()} filas marcadas como no-plausibles (se conservan, solo quedan marcadas)")

Validaciones OK:
  - available_datetime >= timestamp en el 100% de las filas
  - 0 filas sin patient_id, 0 filas sin timestamp
  - 762 filas marcadas como no-plausibles (se conservan, solo quedan marcadas)


## Paso 9 — Guardar Base 1

Se guarda en `../data/processed/timeline_df.csv`, listo para que `02_features_ventanas.ipynb`
lo cargue, arme las ventanas de tiempo y construya Base 2.

In [10]:
os.makedirs(f"{DATA_DIR}/../processed", exist_ok=True)
out_path = f"{DATA_DIR}/../processed/timeline_df.csv"
timeline_df.to_csv(out_path, index=False)
print(f"Guardado: {out_path}")
print(f"Filas: {len(timeline_df)} | Columnas: {list(timeline_df.columns)}")

Guardado: ../data/raw/../processed/timeline_df.csv
Filas: 2535902 | Columnas: ['patient_id', 'timestamp', 'available_datetime', 'variable_code', 'value', 'unit', 'quality_flag', 'plausible', 'source_table', 'record_id']


## Notas y decisiones a declarar en el README

- **`quality_flag` mezcla tipos entre fuentes**: en `vital_signs`/`laboratory_results` es
  categórico (`"OK"`, `"SUSPECT"`, ...), en `device_observations` es un número 0–1
  (`signal_quality`). Se guardan tal cual en la misma columna por ahora — la reconciliación
  a un solo indicador de calidad se hace en Base 2 (notebook 02), no aquí.
- **La deduplicación de retransmisiones solo se aplicó a `vital_signs`**, porque es donde el
  EDA confirmó el patrón (`MONITOR_RETRANSMIT`). `wearable_observations` no tiene columna
  `source_system`, así que no aplica. Si el equipo encuentra el mismo patrón en
  `device_observations`, agregar la misma función ahí.
- **`available_datetime`** se definió como `timestamp` para vital_signs/wearable/device
  (se asume que el dato está disponible apenas se mide) y como `result_datetime` para
  laboratorio (el resultado tarda en procesarse). Esto es una decisión de diseño explícita,
  no un dato que venga en RISA — declarar en el README.
- Este notebook **no borra ninguna fila** por calidad — todo se marca (`plausible`,
  `quality_flag`) y se decide qué hacer con eso en Base 2 (supresión, atenuación). Mantener
  la trazabilidad completa es un requisito de la rúbrica.
